# 03 — Feature Engineering

**Project:** Australian Retail Trade Performance Dashboard  
**Input:** `data/processed/retail_clean.csv`  
**Output:** Three analytical tables consumed by Power BI

---

## Purpose

The cleaned dataset (`retail_clean.csv`) contains raw monthly turnover figures.
To answer the business question — *which categories are underperforming the national benchmark* —
we need calculated fields that Power BI cannot efficiently compute itself:
rolling 12-month totals, year-on-year growth rates, and the gap between each category
and the Total (Industry) benchmark.

This notebook produces three tables, each at a different grain:

| Table | Grain | Joins to | Purpose |
|---|---|---|---|
| `retail_yoy_growth.csv` | date × category | `retail_clean` on date + category | Rolling YoY growth per category over time |
| `retail_benchmark.csv` | date | `retail_clean` on date | National Total Industry benchmark series |
| `retail_category_ranking.csv` | category | `retail_yoy_growth` on category | Snapshot of current underperformer ranking |
| `dim_date.csv` | date | all time-series tables on date | Date dimension for Power BI slicers |

**Why rolling 12-month windows?**  
The EDA (notebook 02) showed December is 2.5–3x a typical month for Department stores and Clothing.
Monthly YoY comparisons would be dominated by the timing of Christmas rather than underlying growth.
Rolling 12-month totals capture one full seasonal cycle per window, making comparisons fair
regardless of which month the window ends on.

### Import libra## Setupries and create path to dataset

In [2]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT   = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

# Load the clean dataset and sort for rolling calculations
df_all = pd.read_csv(DATA_PROCESSED / 'retail_clean.csv', parse_dates=['date'])
df_all = df_all.sort_values(['category', 'date'])

# Split into two dataframes:
# df_total — Total (Industry) only, used as the national benchmark
# df       — the 6 individual categories, used for all category-level analysis
# We split here so the benchmark is always available without re-filtering
df_total = df_all[df_all['category'] == 'Total (Industry)'].copy()
df       = df_all[df_all['category'] != 'Total (Industry)'].copy()

print(f'Working dataset (6 categories): {df.shape}')
print(f'Benchmark dataset (Total Industry): {df_total.shape}')
print(f'Date range: {df["date"].min().strftime("%b %Y")} to {df["date"].max().strftime("%b %Y")}')

Working dataset (6 categories): (3114, 6)
Benchmark dataset (Total Industry): (519, 6)
Date range: Apr 1982 to Jun 2025


## Table 1 — retail_yoy_growth.csv

**Grain:** one row per (date × category) — same as `retail_clean.csv`  
**Joins to:** `retail_clean` on `date + category`

Adds three calculated columns to the 6-category dataset:

- `r12m_turnover` — sum of the current month and the 11 months before it (rolling 12-month total)
- `r12m_prior` — the same rolling total shifted back 12 months (the comparison period)
- `yoy_growth_pct` — percentage change between the two: `(r12m_turnover - r12m_prior) / r12m_prior × 100`

The first 23 rows per category will be `NaN` — 12 months to fill `r12m_turnover`,
then another 12 months shifted back to fill `r12m_prior`. Data from April 1984 onwards is valid.

In [3]:
# Rolling 12-month total — sums the current month + 11 prior months per category
# min_periods=12 ensures we never calculate on a partial window
df['r12m_turnover'] = (
    df.groupby('category')['turnover_m']
    .transform(lambda x: x.rolling(12, min_periods=12).sum())
)

# Shift the rolling total back 12 months to get the prior year comparison period
# e.g. for Jun 2025, r12m_prior = the rolling total that ended Jun 2024
df['r12m_prior'] = (
    df.groupby('category')['r12m_turnover']
    .shift(12)
)

# YoY growth = (current 12M - prior 12M) / prior 12M x 100
df['yoy_growth_pct'] = (
    (df['r12m_turnover'] - df['r12m_prior']) / df['r12m_prior'] * 100
).round(1)

df.to_csv(DATA_PROCESSED / 'retail_yoy_growth.csv', index=False)
print('Saved: retail_yoy_growth.csv')
print(f'Shape: {df.shape}')
print(f'\nLatest values (Jun 2025):')
print(
    df[df['date'] == df['date'].max()]
    [['category', 'r12m_turnover', 'yoy_growth_pct']]
    .sort_values('yoy_growth_pct')
    .to_string(index=False)
)

Saved: retail_yoy_growth.csv
Shape: (3114, 9)

Latest values (Jun 2025):
                                           category  r12m_turnover  yoy_growth_pct
                                  Department stores        23142.0             1.6
      Cafes, restaurants and takeaway food services        66273.2             2.5
Clothing, footwear and personal accessory retailing        36671.7             2.6
                          Household goods retailing        71384.3             3.0
                                     Food retailing       176195.8             3.1
                                    Other retailing        70064.1             6.1


## Table 2 — retail_benchmark.csv

**Grain:** one row per date — no category column  
**Joins to:** `retail_clean` on `date`

Applies the same rolling YoY calculation to the **Total (Industry)** series.
This is the national benchmark — the growth rate that every individual category
is compared against in the ranking and in Power BI.

Keeping it as a separate table (rather than a column on `retail_yoy_growth`) reflects
that it operates at a different grain: one row per month, no category dimension.

In [4]:
df_total = df_total.sort_values('date')

df_total['r12m_turnover'] = df_total['turnover_m'].rolling(12, min_periods=12).sum()
df_total['r12m_prior']    = df_total['r12m_turnover'].shift(12)
df_total['yoy_growth_pct'] = (
    (df_total['r12m_turnover'] - df_total['r12m_prior']) / df_total['r12m_prior'] * 100
).round(1)

# Keep only the columns Power BI needs — no category column
retail_benchmark = df_total[['date', 'turnover_m', 'r12m_turnover', 'yoy_growth_pct']].copy()

retail_benchmark.to_csv(DATA_PROCESSED / 'retail_benchmark.csv', index=False)
print('Saved: retail_benchmark.csv')
print(f'Shape: {retail_benchmark.shape}')
print(f'\nLatest benchmark value:')
print(retail_benchmark[retail_benchmark['yoy_growth_pct'].notna()].tail(3).to_string(index=False))

Saved: retail_benchmark.csv
Shape: (519, 4)

Latest benchmark value:
      date  turnover_m  r12m_turnover  yoy_growth_pct
2025-04-01     35382.3       440693.5             3.0
2025-05-01     36907.1       442145.2             3.1
2025-06-01     36351.9       443730.9             3.3


## Table 3 — retail_category_ranking.csv

**Grain:** one row per category — no date column  
**Joins to:** `retail_yoy_growth` on `category`

A point-in-time snapshot of each category's YoY growth relative to the national benchmark,
anchored to the most recent month in the dataset.
This is the table that directly answers the CFO question:
*which categories are underperforming the national benchmark, and by how many percentage points?*

- `yoy_growth_pct` — the category's rolling 12-month YoY growth at the snapshot date
- `benchmark_growth_pct` — Total Industry YoY growth at the same date (same value for all rows)
- `gap_vs_national_ppt` — the difference: negative = underperforming, positive = outperforming

In [5]:
# Snapshot date — most recent month in the dataset
snapshot_date = df['date'].max()
print(f'Snapshot date: {snapshot_date.strftime("%b %Y")}')

# Pull benchmark value at the snapshot date
benchmark = retail_benchmark[
    retail_benchmark['date'] == snapshot_date
]['yoy_growth_pct'].values[0]
print(f'National benchmark (Total Industry YoY): {benchmark}%')

# Get each category's YoY growth at the snapshot date
latest = df[df['date'] == snapshot_date].copy()
latest['benchmark_growth_pct'] = benchmark
latest['gap_vs_national_ppt']  = (latest['yoy_growth_pct'] - benchmark).round(1)

# Sort worst performers first
retail_category_ranking = (
    latest[['category', 'yoy_growth_pct', 'benchmark_growth_pct', 'gap_vs_national_ppt']]
    .sort_values('yoy_growth_pct')
    .copy()
)

retail_category_ranking.to_csv(DATA_PROCESSED / 'retail_category_ranking.csv', index=False)
print('\nSaved: retail_category_ranking.csv')
print(retail_category_ranking.to_string(index=False))

Snapshot date: Jun 2025
National benchmark (Total Industry YoY): 3.3%

Saved: retail_category_ranking.csv
                                           category  yoy_growth_pct  benchmark_growth_pct  gap_vs_national_ppt
                                  Department stores             1.6                   3.3                 -1.7
      Cafes, restaurants and takeaway food services             2.5                   3.3                 -0.8
Clothing, footwear and personal accessory retailing             2.6                   3.3                 -0.7
                          Household goods retailing             3.0                   3.3                 -0.3
                                     Food retailing             3.1                   3.3                 -0.2
                                    Other retailing             6.1                   3.3                  2.8


## Table 4 — dim_date.csv

**Grain:** one row per month  
**Joins to:** all time-series tables on `date`

A date dimension table that Power BI uses to filter all connected tables simultaneously
through a single year or month slicer. Without this, a slicer on `year` from one table
does not automatically filter other tables that connect to `dim_date` through separate relationships.

Covers the full date range of `retail_clean.csv` — April 1982 to June 2025.

In [6]:
df_clean = pd.read_csv(DATA_PROCESSED / 'retail_clean.csv', parse_dates=['date'])

# One row per month from the first to the last date in the dataset
# freq='MS' = month start — matches the date format in all other tables
dim_date = pd.DataFrame({
    'date': pd.date_range(df_clean['date'].min(), df_clean['date'].max(), freq='MS')
})

dim_date['year']       = dim_date['date'].dt.year
dim_date['month']      = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.strftime('%b')
dim_date['quarter']    = dim_date['date'].dt.quarter

# Australian financial year: FY2024 = Jul 2023 to Jun 2024
dim_date['fy_year']  = dim_date['date'].apply(
    lambda d: d.year if d.month >= 7 else d.year - 1
)
dim_date['fy_label'] = dim_date['fy_year'].apply(
    lambda y: f"FY{str(y)[2:]}/{str(y+1)[2:]}"
)

dim_date.to_csv(DATA_PROCESSED / 'dim_date.csv', index=False)
print('Saved: dim_date.csv')
print(f'Shape: {dim_date.shape}')
print(f'\nSample rows:')
print(dim_date.tail(3).to_string(index=False))

Saved: dim_date.csv
Shape: (519, 7)

Sample rows:
      date  year  month month_name  quarter  fy_year fy_label
2025-04-01  2025      4        Apr        2     2024  FY24/25
2025-05-01  2025      5        May        2     2024  FY24/25
2025-06-01  2025      6        Jun        2     2024  FY24/25
